# പാഠം 03 - ഏജൻസിക് ഡിസൈൻ പാറ്റേണുകൾ

ഈ പാഠത്തിൽ, ഫലപ്രദമായ AI ഏജന്റുകൾ രൂപപ്പെടുത്തുന്നതിനുള്ള മൂന്ന് അടിസ്ഥാനപരമായ ഡിസൈൻ പാറ്റേണുകൾ പരിശോധിക്കുന്നു:

1. **സ്പഷ്ടമായ ഏജന്റ് നിർദ്ദേശങ്ങൾ** — ഏജന്റിന്റെ പെരുമാറ്റം നയിക്കുന്ന, കൃത്യമായ, ഭുമിക നിര്‍വചിക്കുന്ന പ്രോംപ്റ്റുകൾ രൂപകൽപ്പന ചെയ്യൽ  
2. **Pydantic മോഡലുകളുള്ള ഘടനയുള്ള ഔട്ട്പുട്ട്** — ഏജന്റുകൾ പ്രവചിക്കാവുന്ന, സാധൂകരിക്കപ്പെട്ട ഡാറ്റ തിരിച്ചറിയുന്നതിൽ ഉറപ്പ് നൽകൽ  
3. **ഒരൊറ്റ ഉത്തരവാദിത്തമുള്ള ഏജന്റുകൾ** — ഓരോ ഏജന്റും ഒരേ കാര്യം നല്ലതായി ചെയ്യുന്നവിധം പ്രത്യേകം രൂപകൽപ്പന നടത്തൽ  

നാം ഓരോ പാറ്റേണും **പ്രയാണത്തിന്റെ ലക്ഷ്യസ്ഥാന ശുപാർശകർ** എന്ന സാഹചര്യത്തിലേക്ക് പ്രയോഗിക്കും, ഒരുക്കിയിരിക്കുന്ന സംവിധാനം ക്രമേന ലക്ഷ്യസ്ഥാനങ്ങൾ ശുപാർശ ചെയ്യുകയും ലഭ്യത പരിശോധിക്കുകയും ലാജിസ്റ്റിക്സ് കൈകാര്യം ചെയ്യുകയും ചെയ്യാനുള്ള സംവിധാനം നിർമ്മിക്കുന്നു.


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: വ്യക്തമാക്കിയ ഏജന്റ് നിർദ്ദേശങ്ങൾ

ഏറ്റവും പ്രഭാവവത്തായ പാറ്റേൺ ഏറ്റവും ലളിതവുമാണ്: നിങ്ങളുടെ ഏജന്റിന് വ്യക്തമാകുന്ന, വിശദമായ നിർദ്ദേശങ്ങൾ എഴുതുക.

നല്ല നിർദ്ദേശങ്ങൾ നിർവ്വചിക്കുന്നു:
- **ആരാണ്** ഏജന്റ് (പേഴ്‌സോണയും ശൈലിയും)
- **എന്താണ്** ചെയ്യേണ്ടത് (പദപടിയുള്ള ഉത്തരവാദിത്തങ്ങൾ)
- **എങ്ങനെ** പെരുമാറണം (പരിമിതികളും ശൈലിയും)

താഴെ, ഓരോ പ്രതികരണത്തെയും രൂപപ്പെടുത്തുന്ന വ്യക്തമായ നിർദ്ദേശങ്ങളുള്ള ഒരു യാത്രാ കൺസിയർജ് ഏജന്റ് സൃഷ്ടിക്കുന്നു.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic മോഡലുകളുമായി ഘടനാപരമായ ഔട്ട്പുട്ട്

സ്വതന്ത്ര രൂപത്തിലുള്ള എഴുത്ത് സംഭാഷണത്തിന് ഉപകരിക്കുന്നു, പക്ഷേ താഴെ വരെയുള്ള സിസ്റ്റങ്ങൾ ഘടനാപരമായ ഡാറ്റ ആവശ്യപ്പെടുന്നു.  
**Pydantic മോഡലുകൾ** ഒരു **ടൂൾ ഫംഗ്‌ഷനുമായി** കൂട്ടിച്ചേർത്തു, നാം ചെയ്യുക:

- ഏജന്റിന്റെ ഔട്ട്പുട്ടിനായി കൃത്യമായ സ്കീമ നിർവചിക്കുക  
- സ്വയംപ്രകാരമുള്ള റിപ്പോർത്തുകൾ തിരിച്ചറിയുക  
- ഏജന്റ് ഫലങ്ങൾ പ്രയോഗ ലജിക്‌ക്കായി വിശ്വസനീയമായി സംയോജിപ്പിക്കുക  

നാം ഒരു ടൂൾ കൂടി അവതരിപ്പിക്കുന്നു, അത് ലക്ഷ്യസ്ഥല വിശദാംശങ്ങൾ തിരികെ നൽകുന്നതിലൂടെ ഏജന്റ് നിർദ്ദേശങ്ങൾ യഥാർത്ഥ ഡാറ്റയിൽ അടിസ്ഥാനമാക്കുന്നു.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## പാറ്റേൺ 3: ഏകദഫ്രത Agents

സങ്കീർണ്ണമായ ജോലികൾ ഒരുകാരണത്തിലുള്ള ഓരോ ഏജന്റുകളുമായി പല വ്യാപാരത്തെ വേർതിരിച്ച് നടത്തുന്നതിൽ നിന്നു പ്രയോജനം ലഭിക്കുന്നു:

- സ്ഥലങ്ങളും ലഭ്യതയും അറിയുന്ന **ലക്ഷ്യ വിദഗ്ധൻ**
- ഫ്‌ലൈറ്റുകൾ, ഹോട്ടലുകൾ, യാത്ര പ്രണാളികൾ കൈകാര്യം ചെയ്യുന്ന **ലോജിസ്റ്റിക്‌സ് പ്ലാനർ**

ഇത് സോഫ്റ്റ്‌വെയർ എഞ്ചിനീയറിംഗ് തത്വമായ *ഓറിയന്റേഷൻ വേർതിരിക്കൽ* എന്നതിന്റെ പ്രതിബിംബമാണ് — ഓരോ ഏജന്റും സ്വതന്ത്രമായി പരീക്ഷിക്കുക, പരിപാലിക്കുക, മെച്ചപ്പെടുത്തുക എളുപ്പമാണ്.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## സംഗ്രഹം

ഈ പാഠത്തിൽ ഞങ്ങൾ റavel recommender സാഹചര്യത്തിൽ മൂന്ന് ഏജന്റിക് ഡിസൈൻ പാറ്റേണുകൾ പ്രയോഗിച്ചു:

| പാറ്റേൺ | മുഖ്യ ആശയം | ഗുണം |
|---|---|---|
| **സ്പഷ്ടമായ നിർദ്ദേശങ്ങൾ** | വ്യക്തിത്വം, ഉത്തരവാദിത്വങ്ങൾ, നിർബന്ധങ്ങൾ മുൻകൂട്ടി നിർവചിക്കുക | ഒരുപോലെ, ബ്രാൻഡിന്റെ നിർദേശത്തിനു അനുയോജ്യമായ ഏജന്റിന്റെ പെരുമാറ്റം |
| **സമയോജിതമായ ഔട്ട്‌പുട്ട്** | പ്രതികരണ ഫോർമാറ്റായി Pydantic മോടലുകൾ ഉപയോഗിക്കുക | സാധൂകരിച്ച, മെഷീൻ-വായനക്ഷമമായ ഫലങ്ങൾ |
| **ഒറ്റ ഉത്തരവാദിത്വം** | ഓരോ ഏജന്റിനും ഒരു ലക്ഷ്യബദ്ധമായ ജോലി നൽകുക | പരീക്ഷിക്കാൻ, പരിപാലിക്കാൻ, സംയോജിപ്പിക്കാൻ എളുപ്പം |

ഈ പാറ്റേണുകൾ സ്വാഭാവികമായി സംയോജിപ്പിക്കാനാകുന്നു — ഒരൊറ്റ ഉത്തരവാദിത്വ ഏജന്റിനുള്ളിൽ സ്പഷ്ടമായ നിർദ്ദേശങ്ങളും, സമയംോജിതമായ ഔട്ട്‌പുട്ടും ചേർത്ത് പ്രൊഡക്ഷൻ-സജ്ജമായ ശക്തമായ സിസ്റ്റങ്ങൾ നിർമ്മിക്കാൻ കഴിയും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
